# Train your voice for Voice Changer

This notebook turns the dataset ZIP made by **Voice Changer → Clone** into an RVC v2 voice, using the free GPU of Google Colab and [Applio](https://github.com/IAHispano/Applio) 3.7.1 as the trainer.

1. **Runtime → Change runtime type → T4 GPU**, then save.
2. Run the four steps below in order (the ▶ button on each). Keep this tab open while it trains.
3. Step 4 downloads `<name>-rvc.zip`. In Voice Changer, open **Clone → Import trained voice** and choose it.

If Colab disconnects during training, run steps 1–3 again with **keep_in_drive** on: training continues from the last save.

By using this notebook you accept Applio's [terms of use](https://github.com/IAHispano/Applio/blob/main/TERMS_OF_USE.md). Only train voices you have the right to use, such as your own.

In [ ]:
#@title 1. Install the trainer (about 5 minutes)
APPLIO_VERSION = "3.7.1"
APPLIO = "/content/Applio"
PYTHON = f"{APPLIO}/.venv/bin/python"

import os, subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Choose Runtime > Change runtime type > T4 GPU, then run this step again.")
print(gpu.stdout)

if not os.path.exists(PYTHON):
    %cd /content
    !rm -rf "{APPLIO}"
    !git -c advice.detachedHead=false clone -q --depth 1 --branch "{APPLIO_VERSION}" https://github.com/IAHispano/Applio "{APPLIO}"
    %cd "{APPLIO}"
    !apt-get -qq update > /dev/null && apt-get -qq install -y portaudio19-dev > /dev/null
    !curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
    !uv venv .venv --python 3.12 --seed -q
    !uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python "{PYTHON}"
    !"{PYTHON}" core.py prerequisites --models --pretraineds-hifigan --no-exe
    !cp assets/config_template.json assets/config.json
if not os.path.exists(PYTHON):
    raise SystemExit("The trainer did not install; see the messages above.")
%cd "{APPLIO}"
print("Trainer ready.")

In [ ]:
#@title 2. Upload the dataset ZIP
#@markdown Leave **dataset_zip** empty to choose the file from your computer, or give its path in Google Drive (for example `/content/drive/MyDrive/my-voice.zip`).
dataset_zip = ""  #@param {type:"string"}

import json, os, shutil, zipfile
from google.colab import files

if dataset_zip.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not dataset_zip:
    os.chdir("/content")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file chosen.")
    dataset_zip = os.path.join("/content", next(iter(uploaded)))
if not os.path.isfile(dataset_zip):
    raise SystemExit(f"Not found: {dataset_zip}")

DATASET = "/content/dataset"
shutil.rmtree(DATASET, ignore_errors=True)
with zipfile.ZipFile(dataset_zip) as archive:
    archive.extractall(DATASET)
manifest = os.path.join(DATASET, "voice-changer-dataset.json")
if not os.path.isfile(manifest):
    raise SystemExit("This ZIP was not made by Voice Changer (Clone > Pack for Colab).")
info = json.load(open(manifest, encoding="utf-8"))
model_name = info["name"]
takes = len([f for f in os.listdir(f"{DATASET}/dataset") if f.lower().endswith(".wav")])
print(f"Voice '{model_name}': {takes} takes, {info['seconds'] / 60:.1f} minutes.")

In [ ]:
#@title 3. Train (about 1–3 hours)
#@markdown More epochs sound closer to you but take longer; 300 suits 10–20 minutes of recording on a T4.
epochs = 300  #@param {type:"slider", min:50, max:1000, step:10}
batch_size = 8  #@param {type:"slider", min:2, max:16, step:1}
sample_rate = "40k"  #@param ["40k", "48k"]
#@markdown Keep the training in Google Drive (`MyDrive/VoiceChanger/<name>`), so a disconnected session can continue where it stopped.
keep_in_drive = True  #@param {type:"boolean"}

import glob, os
os.chdir(APPLIO)
sr = int(sample_rate.rstrip("k")) * 1000
logs = f"{APPLIO}/logs/{model_name}"
if keep_in_drive:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    kept = f"/content/drive/MyDrive/VoiceChanger/{model_name}"
    os.makedirs(kept, exist_ok=True)
    os.makedirs(f"{APPLIO}/logs", exist_ok=True)
    if not os.path.islink(logs):
        if os.path.isdir(logs):
            !cp -rn "{logs}/." "{kept}/" && rm -rf "{logs}"
        os.symlink(kept, logs)

resuming = bool(glob.glob(f"{logs}/G_*.pth"))
if resuming:
    print("Continuing the earlier training.")
else:
    !"{PYTHON}" core.py preprocess --model-name "{model_name}" --dataset-path "{DATASET}/dataset" --sample-rate {sr} --cpu-cores {os.cpu_count()} --cut-preprocess Automatic --chunk-len 3.0 --overlap-len 0.3 --normalization-mode none
    if not glob.glob(f"{logs}/sliced_audios/*.wav"):
        raise SystemExit("Preparing the recordings failed; see the messages above.")
    !"{PYTHON}" core.py extract --model-name "{model_name}" --f0-method rmvpe --sample-rate {sr} --cpu-cores {os.cpu_count()} --gpu 0 --embedder-model contentvec --include-mutes 2
    if not glob.glob(f"{logs}/extracted/*.npy"):
        raise SystemExit("Extracting features failed; see the messages above.")

!"{PYTHON}" core.py train --model-name "{model_name}" --save-every-epoch 10 --total-epoch {epochs} --sample-rate {sr} --batch-size {batch_size} --gpu 0 --vocoder HiFi-GAN --pretrained --save-only-latest
if not glob.glob(f"{logs}/{model_name}_*e_*s.pth"):
    raise SystemExit("Training stopped before the voice was saved; run this step again to continue.")
print("Training finished. Run step 4.")

In [ ]:
#@title 4. Download the voice
import glob, os, zipfile
from google.colab import files

os.chdir(APPLIO)
logs = f"{APPLIO}/logs/{model_name}"
weights = sorted(glob.glob(f"{logs}/{model_name}_*e_*s.pth"), key=os.path.getmtime)
if not weights:
    raise SystemExit("No trained voice yet: run step 3 until it finishes.")
index = f"{logs}/{model_name}.index"
if not os.path.isfile(index):
    !"{PYTHON}" core.py index --model-name "{model_name}" --index-algorithm Auto
output = f"/content/{model_name}-rvc.zip"
with zipfile.ZipFile(output, "w") as archive:
    archive.write(weights[-1], f"{model_name}.pth")
    if os.path.isfile(index):
        archive.write(index, f"{model_name}.index")
print(f"From {os.path.basename(weights[-1])}. In Voice Changer: Clone > Import trained voice.")
files.download(output)